# 11 - Sensitivity analyses (Table 5.5)
Comparable (same labels and test rows): S1, S4, S5, S7, S8. Ranking only: S2, S3, S6.

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
if SYN:
    from src import synthetic
    df = synthetic.make(6000)
    if not config.SPLITS_LOCK.exists():                   # mirrors the extras notebook 02 writes
        from src import sensitivity
        sp0 = labels.assign_split(df["published_date"])
        m = sensitivity.kev_retroactive_monthly(df[["cve_id", "kev_date_added"]].dropna(),
                                                df[["cve_id", "published_date"]])
        vb = df[(sp0 == "val") & (df["published_date"] > config.VAL_A_END)]
        labels.write_lock(df, "initial_seal", "synthetic dry run",
                          extra={"s1": sensitivity.s1_record(m, df, sp0),
                                 "conformal_alpha": config.conformal_alpha(int(vb["label_12m"].sum()))})
else:
    df = pd.read_parquet(config.DATA / "dataset_labelled.parquet")
lock = labels.require_lock()
X, y, split = labels.get_xy(df, labels.CLEAN)
split.value_counts()

In [ ]:
from src import models, sensitivity, features
params = io_utils.read_json(config.SELECTION_LOCK)["models"]["tfidf-lr"]["params"]
base = models.tfidf_lr(X, y, split, seed=0, **params)["scores"]["test"]

def entry(Xs, ys, ss):
    sc = models.tfidf_lr(Xs, ys, ss, seed=0, **params)["scores"]["test"]
    common = sc.index.intersection(base.index)
    est, lo, hi = evaluate.bootstrap_ci(y[common], sc[common])
    pb = evaluate.paired_bootstrap(y[common], sc[common], base[common])
    return {"pr_auc": est, "ci_lo": lo, "ci_hi": hi, "delta": pb["delta"], "delta_lo": pb["lo"],
            "delta_hi": pb["hi"], "n_pos": int(y[common].sum()), "per_model": pd.Series({"tfidf-lr": est})}

res = {"clean": {"per_model": pd.Series({"tfidf-lr": evaluate.metrics(y[base.index], base)["pr_auc"]})}}
d = df.loc[X.index]
keep1 = ~((split == "train") & (d["published_date"] < pd.Timestamp(lock["s1"]["start"])))
res["S1"] = entry(X[keep1], y[keep1], split[keep1])
res["S7"] = entry(features.mask_frame(X, df), y, split)
keep8 = sensitivity.embargo_mask(d, split)
res["S8"] = entry(X[keep8], y[keep8], split[keep8])
t = sensitivity.summary_table(res)
evaluate.save_table(t, "table_5_5_sensitivity", is_synthetic=SYN)
t

S5: text models already exclude rows without desc_original, so S5 measures the selection
effect with a score that needs no text (EPSS): PR-AUC on all labelled test rows vs restored rows only.

In [ ]:
lab = df[df["label_12m"].notna() & (labels.assign_split(df["published_date"]) == "test")
         & (df["is_zero_day"] == 0) & df["epss_at_pred"].notna()]
r = lab["desc_original"].notna()
s5 = pd.DataFrame({"all_rows": evaluate.metrics(lab["label_12m"], lab["epss_at_pred"]),
                   "restored_only": evaluate.metrics(lab.loc[r, "label_12m"], lab.loc[r, "epss_at_pred"])}).T
evaluate.save_table(s5, "table_s5_selection_effect", is_synthetic=SYN)
s5[["n", "n_pos", "base_rate", "pr_auc", "lift"]]

S2 (`label="label_any_kev"` in get_xy), S3 (`labels.relabel_horizon`), S4 (rebuild with
`T_PRED_OFFSET_DAYS = 7` and ZERO_DAY_DAYS >= 7), S6 (`data.kev_union`) follow the same pattern;
add every model of the ladder to `per_model` so Kendall tau compares rankings.